# Select the Firms and Filings

| Item | Description |
|---|---|
| Goal | Fix the firm universe and the two filing samples that every later notebook of T3 reads. The universe is the S&P 500 firms with a usable 10-K Item 1 (Business) section, one filing per firm and fiscal year. |
| Input | `sp500_10k_item1.parquet`, one row per 10-K filing with its Item 1 text and a usability flag, and `sample_firms.parquet`, the firm list with the S&P 500 flag. Both come from D3, in its Processed folder on the scratch root. |
| Output | `filings.parquet` in the T3.1 Processed folder: one row per kept filing with its sample flags. `table_sample_filings.csv` and `table_filings_by_fiscal_year.csv` under `R - Results/tables/...`, the same list and its coverage by year. |
| Variables | • `in_all_years`: 1 for every kept filing, the all-years sample<br>• `in_three_years`: 1 if the filing's fiscal year is 2010, 2015, or 2020<br>• `n_three_years`: the number of the three fiscal years in which the firm has a kept filing, 0 to 3, repeated on each of the firm's rows |
| Run | Run 2026-10-31. The list equals the one of the October 2026 run that the production steps read: 18,779 filings of 1,058 firms. |


## Step 1. Import packages


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display


## Step 2. Define file locations


In [2]:
PROJECT_FOLDER = Path.cwd().resolve().parents[2]  # <pipeline>/<section>/<subfolder>/ sits three levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("T - Three Step Text Analysis Pipeline/T3 - Topic Relevance and Corporate Disclosure/T3.1 - Sample - Firms and 10-K Item 1 Filings")
NOTEBOOK_NAME = "T3.1.1 - Sample - Select the Firms and Filings"  # the notebook filename without .ipynb
RAW_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Raw"
INTERIM_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Interim"
PROCESSED_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Processed"
INPUT_FOLDER = SCRATCH_FOLDER / "D - Data Collection and Processing/D3 - SEC Form 10-K Corporate Filings" / "Processed"
ITEM1_FILE = INPUT_FOLDER / "sp500_10k_item1.parquet"
FIRMS_FILE = INPUT_FOLDER / "sample_firms.parquet"

YEAR_MIN = 1999
YEAR_MAX = 2025
THREE_YEARS = [2010, 2015, 2020]
FIGURE_FOLDER = PROJECT_FOLDER / "R - Results/figures" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
TABLE_FOLDER = PROJECT_FOLDER / "R - Results/tables" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
MODEL_CACHE = SCRATCH_FOLDER / "cache" / "huggingface"
ENV_FILE = PROJECT_FOLDER / "T - Three Step Text Analysis Pipeline/T3 - Topic Relevance and Corporate Disclosure" / ".env"
for folder in (PROCESSED_FOLDER, TABLE_FOLDER):
    folder.mkdir(parents=True, exist_ok=True)


## Step 3. Load the filing table

Every filing's identifiers, fiscal year, filing date, form type, and usability flag. The Item 1 text itself is not needed here.


In [3]:
filings = pd.read_parquet(ITEM1_FILE, columns=["accessionNo", "cik", "gvkey", "conm", "fyear", "periodEnd", "filedAt", "formType", "usable"])
assert filings["accessionNo"].is_unique
assert filings[["gvkey", "fyear", "filedAt"]].notna().all().all()
print(f"Filings in the source table: {len(filings):,}; firms: {filings['gvkey'].nunique():,}")
print(f"Form types: {filings['formType'].value_counts().to_dict()}")
display(filings.head(3))


Filings in the source table: 22,299; firms: 1,061
Form types: {'10-K': 20539, '10-K405': 1739, '10-KT': 21}


,accessionNo,cik,gvkey,conm,fyear,periodEnd,filedAt,formType,usable
0,0000912057-94-000081,61478,001013,ADC TELECOMMUNICATIONS INC,1993,1993-10-31,1994-01-12 05:00:00+00:00,10-K,True
1,0000912057-94-004277,61478,001013,ADC TELECOMMUNICATIONS INC,1994,1994-10-31,1994-12-22 05:00:00+00:00,10-K,True
2,0000061478-96-000002,61478,001013,ADC TELECOMMUNICATIONS INC,1995,1995-10-31,1996-01-05 05:00:00+00:00,10-K,True


## Step 4. Load the firm list

One row per firm (gvkey) with its S&P 500 flag. Only the flag is used here.


In [4]:
firms = pd.read_parquet(FIRMS_FILE, columns=["gvkey", "conm", "sic", "in_sp500"])
assert firms["gvkey"].is_unique
print(f"Firms in the firm list: {len(firms):,}; flagged S&P 500-ever: {int(firms['in_sp500'].sum()):,}")
display(firms.head(3))


Firms in the firm list: 25,207; flagged S&P 500-ever: 1,075


,gvkey,conm,sic,in_sp500
0,007596,MOVIE SYSTEMS INC,4841,False
1,006314,K-TRON INTERNATIONAL INC,3823,False
2,011007,UC CORP,7385,False


## Step 5. Confirm the firm universe

Every firm in the filing table must appear in the firm list with the S&P 500 flag set. The filing table is then the S&P 500-ever universe and needs no further membership filter.


In [5]:
filing_firms = filings[["gvkey"]].drop_duplicates().merge(firms[["gvkey", "in_sp500"]], on="gvkey", how="left", validate="one_to_one")
assert filing_firms["in_sp500"].notna().all()
assert filing_firms["in_sp500"].all()
print(f"Firms in the filing table: {len(filing_firms):,}; all flagged S&P 500-ever.")


Firms in the filing table: 1,061; all flagged S&P 500-ever.


## Step 6. Keep usable filings in the sample window

Drop filings whose Item 1 could not be extracted and filings outside fiscal years 1999 to 2025. Each rule reports how many filings it removes.


In [6]:
n_source = len(filings)
not_usable = ~filings["usable"]
outside_window = ~filings["fyear"].between(YEAR_MIN, YEAR_MAX)
filings = filings.loc[~not_usable & ~outside_window].copy()
print(f"Dropped as not usable: {int(not_usable.sum()):,}")
print(f"Dropped as outside {YEAR_MIN}-{YEAR_MAX} (among usable): {int((outside_window & ~not_usable).sum()):,}")
print(f"Kept: {len(filings):,} of {n_source:,} filings; firms: {filings['gvkey'].nunique():,}")


Dropped as not usable: 333
Dropped as outside 1999-2025 (among usable): 3,164
Kept: 18,802 of 22,299 filings; firms: 1,058


## Step 7. Resolve firm-years with two usable filings

A few firm-years carry two usable filings: a full-year report followed by a transition-period report, two reports with the same fiscal-year label, or a same-day duplicate. In every case the earlier-filed filing is the ordinary full-year report, so it is kept. Ties go to the lower accession number.


In [7]:
duplicated_firm_years = filings.loc[filings.duplicated(["gvkey", "fyear"], keep=False)].sort_values(["gvkey", "fyear", "filedAt"])
print(f"Firm-years with two usable filings: {duplicated_firm_years.groupby(['gvkey', 'fyear']).ngroups:,}")
display(duplicated_firm_years[["gvkey", "conm", "fyear", "formType", "periodEnd", "filedAt", "accessionNo"]])

filings = filings.sort_values(["gvkey", "fyear", "filedAt", "accessionNo"]).drop_duplicates(["gvkey", "fyear"], keep="first").copy()
assert not filings.duplicated(["gvkey", "fyear"]).any()
print(f"Filings after the rule: {len(filings):,}; firms: {filings['gvkey'].nunique():,}")


Firm-years with two usable filings: 23


,gvkey,conm,fyear,formType,periodEnd,filedAt,accessionNo
863,001722,ARCHER-DANIELS-MIDLAND CO,2012,10-K,2012-06-30,2012-08-25 01:11:10+00:00,0000007084-12-000034
864,001722,ARCHER-DANIELS-MIDLAND CO,2012,10-KT,2012-12-31,2013-02-28 22:24:04+00:00,0000007084-13-000014
4148,004737,FIRST HORIZON CORP,2000,10-K,2000-12-31,2001-03-28 05:00:00+00:00,0000950144-01-004077
4147,004737,FIRST HORIZON CORP,2000,10-K405,2001-03-31,2002-03-25 05:00:00+00:00,0000950144-02-002671
4862,005492,L3HARRIS TECHNOLOGIES INC,2019,10-K,2019-06-28,2019-08-22 19:04:00+00:00,0000202058-19-000090
4861,005492,L3HARRIS TECHNOLOGIES INC,2019,10-KT,2020-01-03,2020-03-03 21:17:25+00:00,0000202058-20-000011
6102,006669,LENNAR CORP,2004,10-K,2004-11-30,2005-02-12 02:32:09+00:00,0001193125-05-027428
6103,006669,LENNAR CORP,2004,10-K,2005-01-30,2006-02-09 22:34:38+00:00,0001193125-06-025438
6413,006821,LOUISIANA-PACIFIC CORP,2007,10-K,2007-12-31,2008-03-07 15:55:40+00:00,0001047469-08-002334
6414,006821,LOUISIANA-PACIFIC CORP,2007,10-K,2008-01-31,2009-02-27 22:28:41+00:00,0001193125-09-041236


Filings after the rule: 18,779; firms: 1,058


## Step 8. Flag the two samples

The all-years sample is every kept filing. The three-year sample is the filings for fiscal years 2010, 2015, and 2020; a firm stays in it with however many of the three years it has.


In [8]:
filings["in_all_years"] = 1
filings["in_three_years"] = filings["fyear"].isin(THREE_YEARS).astype(int)
three_year_counts = filings.loc[filings["in_three_years"].eq(1)].groupby("gvkey")["fyear"].nunique()
filings["n_three_years"] = filings["gvkey"].map(three_year_counts).fillna(0).astype(int)
print(f"All-years sample: {len(filings):,} filings, {filings['gvkey'].nunique():,} firms")
print(f"Three-year sample: {int(filings['in_three_years'].sum()):,} filings, {three_year_counts.size:,} firms")
print("Firms by number of the three years present:")
print(three_year_counts.value_counts().sort_index().rename("firms").to_string())


All-years sample: 18,779 filings, 1,058 firms
Three-year sample: 2,072 filings, 841 firms
Firms by number of the three years present:
fyear
1    153
2    145
3    543


## Step 9. Count filings and firms by fiscal year

Coverage of the all-years sample, saved as a table.


In [9]:
coverage = filings.groupby("fyear").agg(filings=("accessionNo", "size"), firms=("gvkey", "nunique"))
assert coverage["filings"].sum() == len(filings)
coverage.to_csv(TABLE_FOLDER / "table_filings_by_fiscal_year.csv")
print(f"Fiscal years covered: {coverage.index.min()}-{coverage.index.max()}")
display(coverage.T)


Fiscal years covered: 1999-2025


fyear,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
filings,703,692,713,726,730,738,732,728,737,725,...,688,678,674,670,654,655,648,649,643,625
firms,703,692,713,726,730,738,732,728,737,725,...,688,678,674,670,654,655,648,649,643,625


## Step 10. Save the filing list

One row per kept filing with the sample flags. Later notebooks filter on `in_all_years` or `in_three_years` and never reapply the rules above.


In [10]:
output_columns = ["accessionNo", "gvkey", "cik", "conm", "fyear", "filedAt", "in_all_years", "in_three_years", "n_three_years"]
filings[output_columns].to_parquet(PROCESSED_FOLDER / "filings.parquet", index=False)
filings[output_columns].to_csv(TABLE_FOLDER / "table_sample_filings.csv", index=False)
print(f"Saved filings.parquet and table_sample_filings.csv: {len(filings):,} rows; columns {output_columns}")


Saved filings.parquet and table_sample_filings.csv: 18,779 rows; columns ['accessionNo', 'gvkey', 'cik', 'conm', 'fyear', 'filedAt', 'in_all_years', 'in_three_years', 'n_three_years']
